# Lab 2: The forward pass, by hand

**DSAN-6600 Deep Learning · Fall 2026**

**Due:** Wednesday Sep 9, 11:59 PM ET · **Target time:** about 60–75 minutes

---

## What this lab is for

Week 2 argued that a layer is a matrix multiply, that stacking linear layers
collapses, and that you can train a small network without backpropagation, at a
cost that does not scale. This lab has you do all three in NumPy, so that when
PyTorch autograd shows up next week you already know what it is replacing.

No PyTorch. Just the arithmetic a network actually does.

## How this lab is graded

- **The notebook is graded for completion, not for correctness.** Submit it,
  with all cells run and outputs visible, and you get the points.
- **You may work together.** Everyone runs the same seed, so your numbers
  *should* match your neighbor's.
- **The understanding is assessed on the Week 3 quiz.** Roughly a third of that
  quiz asks you to explain **what this lab demonstrated and why**.
- **Solutions are posted after the deadline.**

## Using AI on this lab

Allowed and expected, with disclosure; there is a cell for that at the end. An
agent can write this code in a minute. It cannot sit Thursday's quiz for you.

## Step 0: Setup

Everyone in the class runs the **same seed**. It is the same `6600` the lecture
used, so the numbers you print should match the ones on those slides.

In [1]:
SEED = 6600      # the whole class uses this, so everyone sees the same data

import sys

import numpy as np
import matplotlib.pyplot as plt

print(f"python  {sys.version.split()[0]}")
print(f"numpy   {np.__version__}")
print(f"seed    {SEED}")

python  3.13.15
numpy   2.1.3
seed    6600


---

# Part 1 · A layer as a matrix multiply

A layer with $n_{\text{out}}$ units and $n_{\text{in}}$ inputs is:

$$\mathbf{z} = \mathbf{W}\mathbf{x} + \mathbf{b}$$

$\mathbf{W}$ has shape $(n_{\text{out}},\, n_{\text{in}})$: **rows are units,
columns are inputs.** That is the same convention as PyTorch's `nn.Linear` and
as Thursday's lecture.

Fill in `linear` and `n_params`. The guard will catch a missing return.

In [2]:
def linear(x, W, b):
    """One layer: z = W x + b.
    x : (n_in,)
    W : (n_out, n_in)
    b : (n_out,)
    returns z : (n_out,)
    """
    z = W @ x + b
    return z

def n_params(W, b):
    """How many numbers in this layer get trained?"""
    n = W.size + b.size
    return n


rng = np.random.default_rng(SEED)
W = rng.normal(0, 0.8, (3, 4))
b = np.zeros(3)
x = rng.normal(0, 1, 4)

z = linear(x, W, b)
print(f"x  {x.shape}   W  {W.shape}   b  {b.shape}   z  {z.shape}")
print(f"parameters in this layer: {n_params(W, b)}")
assert z.shape == (3,), f"expected z shape (3,), got {z.shape}"
assert n_params(W, b) == 15, f"3*4 + 3 = 15, got {n_params(W, b)}"
print("shapes and count check out.")

x  (4,)   W  (3, 4)   b  (3,)   z  (3,)
parameters in this layer: 15
shapes and count check out.


**Q1.** A classmate writes `W` with shape $(4, 3)$ for this layer, "because
there are 4 inputs and 3 outputs." What goes wrong, and which of the two numbers
in `(n_out, n_in)` is the one that has to match the incoming vector?

*Your answer:*

---

# Part 2 · The 2-3-2-1 forward pass

Same network as the lecture: two inputs, hidden layers of 3 and 2, one output.
Tanh on the hidden layers, **no** activation on the output (this is scalar
regression).

Weights are already initialized. Fill in the two forward passes: one example,
then a batch.

In [3]:
rng = np.random.default_rng(SEED)

W1, b1 = rng.normal(0, 0.8, (3, 2)), np.zeros(3)
W2, b2 = rng.normal(0, 0.8, (2, 3)), np.zeros(2)
W3, b3 = rng.normal(0, 0.8, (1, 2)), np.zeros(1)

x = np.array([0.7, -1.2])
X = rng.normal(0, 1, (5, 2))     # five examples, two features each
print(f"single example {x.shape}, batch {X.shape}")
print(f"W1 {W1.shape}  W2 {W2.shape}  W3 {W3.shape}")

single example (2,), batch (5, 2)
W1 (3, 2)  W2 (2, 3)  W3 (1, 2)


In [ ]:
def forward_one(x, W1, b1, W2, b2, W3, b3):
    """Single example. x is (2,). Returns a1, a2, y_hat."""
    # TODO: hidden 1 (tanh), hidden 2 (tanh), output (linear).
    a1 = None
    a2 = None
    y_hat = None
    if a1 is None or a2 is None or y_hat is None:
        raise NotImplementedError("forward_one: fill a1, a2, y_hat")
    return a1, a2, y_hat


def forward_batch(X, W1, b1, W2, b2, W3, b3):
    """Batch of examples. X is (N, 2). Returns Y of shape (N, 1).

    Hint from lecture: the batched form is X @ W.T + b, not W @ X.
    """
    # TODO: the same three steps, on a matrix of rows.
    Y = None
    if Y is None:
        raise NotImplementedError("forward_batch: return the (N, 1) predictions")
    return Y


a1, a2, y_hat = forward_one(x, W1, b1, W2, b2, W3, b3)
Y = forward_batch(X, W1, b1, W2, b2, W3, b3)

total = n_params(W1, b1) + n_params(W2, b2) + n_params(W3, b3)
print(f"x      {str(x.shape):>8}   {np.round(x, 3)}")
print(f"a1     {str(a1.shape):>8}   {np.round(a1, 3)}")
print(f"a2     {str(a2.shape):>8}   {np.round(a2, 3)}")
print(f"y_hat  {str(y_hat.shape):>8}   {np.round(y_hat, 3)}")
print(f"X {X.shape} -> Y {Y.shape}")
print(f"parameters: {total}")

assert a1.shape == (3,) and a2.shape == (2,) and y_hat.shape == (1,)
assert Y.shape == (5, 1)
assert total == 20, f"2-3-2-1 has 20 parameters, got {total}"

one_at_a_time = np.array([
    forward_one(xi, W1, b1, W2, b2, W3, b3)[2][0] for xi in X
])
max_diff = float(np.max(np.abs(one_at_a_time - Y.ravel())))
print(f"max difference, batch vs loop: {max_diff:.2e}")
assert max_diff < 1e-12, "batched forward does not match the loop; check the .T"
print("2-3-2-1 checks out.")

**Q2.** The batched version used `X @ W.T`, not `W @ X`. Why does the transpose
show up, given that a single example used `W @ x`?

*Your answer:* The inner dimensions now align because the batched turns x column intro rows, so we have to undo the shape

---

# Part 3 · Linear layers collapse

A $1 \to 8 \to 8 \to 1$ stack with **no activation anywhere**. The lecture
claimed this is secretly a single linear layer. Multiply the matrices out and
check.

Because $\mathbf{W}$ is `(units, inputs)`, the collapse is:

$$\mathbf{W}_{\text{eff}} = \mathbf{W}_3\,\mathbf{W}_2\,\mathbf{W}_1,
  \qquad
  \mathbf{b}_{\text{eff}} = \mathbf{W}_3(\mathbf{W}_2\mathbf{b}_1 + \mathbf{b}_2) + \mathbf{b}_3$$

In [ ]:
rng = np.random.default_rng(SEED)
sizes = [1, 8, 8, 1]
Ws = [rng.normal(0, 1 / np.sqrt(n_in), (n_out, n_in))
      for n_in, n_out in zip(sizes[:-1], sizes[1:])]
bs = [np.zeros(n_out) for n_out in sizes[1:]]
x_line = np.linspace(-1, 1, 200).reshape(-1, 1)
print(f"stack shapes: {[W.shape for W in Ws]}")

In [ ]:
def deep_linear(X, Ws, bs):
    """Forward pass with no activation anywhere. X is (N, 1)."""
    # TODO: apply each (W, b) in order, using the batched form X @ W.T + b.
    a = None
    if a is None:
        raise NotImplementedError("deep_linear")
    return a.ravel()


def collapse(Ws, bs):
    """Multiply the stack into one W_eff (1, 1) and one b_eff (1,)."""
    # TODO: start from the identity and fold each layer in.
    W_eff = None
    b_eff = None
    if W_eff is None or b_eff is None:
        raise NotImplementedError("collapse")
    return W_eff, b_eff


deep = deep_linear(x_line, Ws, bs)
W_eff, b_eff = collapse(Ws, bs)
shallow = (x_line @ W_eff.T + b_eff).ravel()

n_deep = sum(W.size for W in Ws) + sum(b.size for b in bs)
n_shallow = W_eff.size + b_eff.size
gap = float(np.max(np.abs(deep - shallow)))
print(f"three linear layers : {n_deep} parameters")
print(f"the one it equals   : {n_shallow} parameters")
print(f"max difference      : {gap:.2e}")
assert n_deep == 97 and n_shallow == 2
assert gap < 1e-10, "the two forwards should be identical; check the multiply order"
print("collapse checks out.")

The collapse above used one particular set of weights. It is not a coincidence
of that draw: **no** setting of those 97 numbers escapes a straight line.

The cell below draws eight random weight sets and plots what each stack
computes, with and without tanh. Nothing is trained here; this is about the
shape of the functions the architecture can produce at all. To put a number on
it, `bend` fits a straight line to the network's own output and reports the
largest gap from it. Exactly zero means the network *is* a straight line.

In [ ]:
def stack(X, Ws, bs, activation):
    """Forward pass with tanh between layers, or with nothing between them."""
    a = X
    for i, (W, b) in enumerate(zip(Ws, bs)):
        z = a @ W.T + b
        a = z if i == len(Ws) - 1 else (np.tanh(z) if activation else z)
    return a.ravel()


def bend(f):
    """Largest gap between the network's output and the best straight line."""
    straight = np.polyval(np.polyfit(x_line.ravel(), f, 1), x_line.ravel())
    return float(np.max(np.abs(f - straight)))


def random_stack(seed, scale=2.5):
    r = np.random.default_rng(seed)
    return ([r.normal(0, scale / np.sqrt(n_in), (n_out, n_in))
             for n_in, n_out in zip(sizes[:-1], sizes[1:])],
            [r.normal(0, 0.5, n_out) for n_out in sizes[1:]])


fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
print(f"{'draw':>5}{'bend, no activation':>22}{'bend, tanh between':>22}")
for d in range(8):
    Wd, bd = random_stack(SEED + d)
    lin, tan = stack(x_line, Wd, bd, False), stack(x_line, Wd, bd, True)
    axes[0].plot(x_line, lin, lw=1.8, alpha=0.85)
    axes[1].plot(x_line, tan, lw=1.8, alpha=0.85)
    print(f"{d:>5}{bend(lin):>22.1e}{bend(tan):>22.3f}")

axes[0].set_title("97 parameters, no activation")
axes[1].set_title("the same 97, tanh between layers")
for ax in axes:
    ax.set_xlabel("x")
    ax.axhline(0, color="#cccccc", lw=1, zorder=0)
axes[0].set_ylabel("network output")
fig.tight_layout()
plt.show()

**Q3.** Every draw in the left panel came out a straight line, and its `bend`
was zero to machine precision. In one sentence, why can no choice of those 97
numbers produce anything else?

Then: the right panel used **the same 97 numbers**, and none of them are
straight. What did inserting tanh change about the set of functions this
architecture can reach?

*Your answer:*

---

# Part 4 · Training with finite differences

A $1 \to 16 \to 1$ tanh network, fitting a noisy sine. The forward pass and
the unpacking are provided. Your job is the gradient, the training loop, and
the forward-pass count.

The finite-difference recipe, from lecture:

$$\frac{\partial \mathcal{L}}{\partial w_j} \approx
  \frac{\mathcal{L}(w_j + \varepsilon) - \mathcal{L}(w_j)}{\varepsilon}$$

One extra forward pass per parameter, plus one for the un-nudged loss. That is
the number to watch.

In [ ]:
HIDDEN, STEPS, LR, EPS = 16, 400, 0.1, 1e-5
SHAPES = [("W", (HIDDEN, 1)), ("b", (HIDDEN,)), ("W", (1, HIDDEN)), ("b", (1,))]
N_PARAMS = sum(int(np.prod(s)) for _, s in SHAPES)

rng_data, rng_init = np.random.default_rng(SEED), np.random.default_rng(SEED)
x = np.linspace(-1, 1, 200).reshape(-1, 1)
y_true = np.sin(3.0 * x.ravel())
y = y_true + rng_data.normal(0, 0.10, 200)

def unpack(theta):
    out, i = [], 0
    for _, shape in SHAPES:
        n = int(np.prod(shape))
        out.append(theta[i:i + n].reshape(shape))
        i += n
    return out

calls = 0
def forward(X, theta):
    global calls
    calls += 1
    W1, b1, W2, b2 = unpack(theta)
    return (np.tanh(X @ W1.T + b1) @ W2.T + b2).ravel()

def loss(theta):
    return float(np.mean((forward(x, theta) - y) ** 2))

theta0 = np.zeros(N_PARAMS)
i = 0
for kind, shape in SHAPES:
    n = int(np.prod(shape))
    theta0[i:i + n] = (rng_init.normal(0, 1 / np.sqrt(shape[1]), n)
                       if kind == "W" else 0.0)
    i += n

print(f"parameters: {N_PARAMS}")
print(f"data: {x.shape[0]} points, noise sd 0.10")

In [ ]:
def finite_difference_gradient(theta):
    """Return (grad, base_loss).

    One forward pass for the current loss, then one more for each parameter.
    Put the bumped parameter back; do not train on the bumped copy.
    """
    # TODO
    grad = None
    base = None
    if grad is None or base is None:
        raise NotImplementedError("finite_difference_gradient")
    return grad, base


theta = theta0.copy()
calls = 0
history = []

# TODO: STEPS of gradient descent.
#   grad, base = finite_difference_gradient(theta)
#   history.append(base)
#   theta = theta - LR * grad
# After the loop, one more loss(theta) is the final value.
if len(history) < STEPS:
    raise NotImplementedError("training loop: take STEPS steps")

end = loss(theta)
history.append(end)
start = history[0]
straight_line = float(np.mean(
    (np.polyval(np.polyfit(x.ravel(), y, 1), x.ravel()) - y) ** 2))
floor = float(np.mean((y - y_true) ** 2))

print(f"parameters                  {N_PARAMS}")
print(f"forward passes per gradient {N_PARAMS + 1}")
print(f"total forward passes        {calls:,}")
print()
print(f"loss at the start           {start:.4f}")
print(f"loss at the end             {end:.4f}")
print(f"a straight line would give  {straight_line:.4f}")
print(f"the noise floor is          {floor:.4f}")

assert calls == STEPS * (N_PARAMS + 1) + 1, (
    f"expected {STEPS * (N_PARAMS + 1) + 1} forward passes "
    f"(N_PARAMS+1 per step, plus one final loss), got {calls}"
)
assert end < 0.05, f"loss should drop well below 0.05, got {end:.4f}"
assert end < straight_line, "the network should beat a straight line"
print("finite-difference training checks out.")

The fit, and the loss on a log scale. The dashed line on the left is a straight
line through the same points; the dotted line on the right is the noise floor.

In [ ]:
training_passes = calls          # freeze the count before plotting spends more
pred = forward(x, theta)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
axes[0].scatter(x, y, s=10, alpha=0.45, color="#4c78a8", label="data")
axes[0].plot(x, pred, color="#e45756", lw=2.2, label="MLP (finite differences)")
axes[0].plot(x, np.polyval(np.polyfit(x.ravel(), y, 1), x.ravel()),
             color="#54a24b", lw=1.6, ls="--", label="straight line")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y"); axes[0].legend(frameon=False)
axes[0].set_title(f"fit   MSE {end:.4f}")
axes[1].plot(history, color="#e45756", lw=2)
axes[1].axhline(straight_line, color="#54a24b", ls="--", lw=1.4,
                label=f"straight line {straight_line:.3f}")
axes[1].axhline(floor, color="#9d755d", ls=":", lw=1.6,
                label=f"noise floor {floor:.3f}")
axes[1].set_yscale("log")
axes[1].set_xlabel("step"); axes[1].set_ylabel("MSE")
axes[1].legend(frameon=False)
axes[1].set_title(f"{training_passes:,} forward passes")
fig.tight_layout()
plt.show()

**Q4.** Each gradient cost 50 forward passes for a 49-parameter network. Where
does the extra 1 come from? If the network had 25 million parameters (ResNet-50),
how many forward passes would **one** gradient step take?

*Your answer:*

**Q5.** The network beat a straight line and ended near the noise floor, so the
method *works*. Next week's lecture replaces it anyway. In one or two sentences:
what is the method failing to scale with, and why is that fatal for the models
this course actually trains?

*Your answer:*

---

# Part 5 · What this lab showed

**This is the part that gets assessed.** Roughly a third of Quiz 2 asks you to
explain the things below. Nothing to memorize and no numbers to transcribe: the
quiz asks *why*, not *what was your value*.

Scroll back through your own output and make sure you can say, in a sentence each:

1. **Why a layer is a matrix multiply**, and why $\mathbf{W}$ is shaped
   `(units, inputs)` rather than the other way around.
2. **Why the batched forward pass needs a transpose** relative to the
   single-example version, and why the two still agree.
3. **Why a 2-3-2-1 network has exactly 20 parameters.** The pattern per layer.
4. **Why stacked linear layers collapse.** What the extra 95 parameters bought
   you in Part 3, and what inserting tanh changed.
5. **Why finite-difference gradients cost one extra forward pass per
   parameter**, plus one for the un-nudged loss.
6. **Why that cost is fatal** once the model has millions of parameters, even
   though the method produced a perfectly good fit on 49 parameters.
7. **What next week is replacing.** Autograd computes the same gradient; it
   does not take 25 million extra forward passes to do it.

If you can answer those seven, you are ready for the quiz.

---

# Part 6 · AI disclosure

Required if you used any generative AI on this lab. Name the tool, say where you
used it, and say what for. "None" is a perfectly good answer.

*Tool(s):*

*Where:*

*What for:*

---

# Submitting

1. **Restart the kernel and Run All.** Confirm it runs top to bottom.
2. Check that every plot and every printed block is visible.
3. Render to **HTML or PDF** with resources embedded.
4. Submit the rendered file on Canvas. Due **Wed Sep 9, 11:59 PM ET**.

The notebook is graded for completion, so this is mostly a formality. The
understanding is graded on Thursday, closed-book, with no notes. Part 5 is your
study guide for that.